## Summary
Combines model scores and business signals into final merchant rankings. Normalizes inputs, applies weights, computes segment-level ranks, and exports final tables.


In [ ]:
# Imports and settings
from pyspark.sql import SparkSession
from pyspark.sql import Window
from pyspark.sql import functions as F


In [ ]:
spark = (
    SparkSession.builder
    .appName("LocalETL")
    # === MEMORY ALLOCATION ===
    .config("spark.driver.memory", "10g")        # main process (driver)
    .config("spark.executor.memory", "10g")      # worker processes (same JVM locally)
    .config("spark.driver.maxResultSize", "2g")  # prevent result collection errors
    
    # === PARALLELISM ===
    .config("spark.driver.cores", "6")           # use 6 out of 8 cores
    .config("spark.executor.cores", "6")
    .config("spark.default.parallelism", "12")   # usually ~2x num_cores for local
    
    # === PERFORMANCE TUNING ===
    .config("spark.sql.shuffle.partitions", "24")  # parallelize shuffles (joins/groupBy)
    .config("spark.memory.fraction", "0.8")        # 80% of JVM heap for Spark execution
    .config("spark.memory.storageFraction", "0.4") # 40% of that for caching
    .config("spark.sql.files.maxPartitionBytes", "128MB")  # ideal partition size for I/O
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")  # fast Pandas conversion
    .config("spark.local.dir", "/tmp/spark-temp")  # local disk space for shuffle spill
    
    # === OPTIONAL LOGGING + CLEANUP ===
    .config("spark.sql.broadcastTimeout", "600")  # allow large table broadcasts
    .config("spark.cleaner.periodicGC.interval", "5min")  # reduce memory leaks
    
    .getOrCreate()
)


In [ ]:

from pyspark.sql import Window
from pyspark.sql import functions as F

fraud_ranks = spark.read.csv("../data/curated/merchant_fraud_rankings.csv", header=True, inferSchema=True)
growth_ranks = spark.read.csv("../data/curated/merchant_growth_rankings.csv", header=True, inferSchema = True)
merchant_transactions = spark.read.parquet("../data/curated/merchant_transactions")


In [ ]:
fraud_ranks.show(10)

In [ ]:
growth_ranks.show(10)

In [ ]:
merchant_transactions.show(10)
merchant_transactions_agg = merchant_transactions.groupBy('merchant_abn', 'segment', 'business', 'take_rate').agg(
    F.sum('dollar_value').alias('total_revenue'),
    F.min("order_datetime").alias("first_order_date"),
    F.max("order_datetime").alias("last_order_date")

    )

joined_rankings = merchant_transactions_agg.join(
    fraud_ranks, on='merchant_abn', how='left').join(
        growth_ranks, on='merchant_abn', how='left')

joined_rankings = joined_rankings.withColumn(
    "date_range_months",
    F.ceil(F.months_between(F.col("last_order_date"), F.col("first_order_date")))
).drop("first_order_date", "last_order_date")

joined_rankings = joined_rankings.withColumn(
    "avg_monthly_takings",
    F.col('take_rate') / 100 * F.col("total_revenue") / F.col("date_range_months")
)

joined_rankings.orderBy('avg_monthly_takings', ascending=False)

window = Window.orderBy(F.desc("avg_monthly_takings"))

joined_rankings = joined_rankings.withColumn("Takings Rank", F.row_number().over(window))

joined_rankings = joined_rankings.withColumn(
    "loss_potential",
    F.col("avg_monthly_takings") * (F.col("eb_p"))
)
window = Window.orderBy(F.desc("loss_potential"))

joined_rankings = joined_rankings.withColumn("Loss Potential Ranking", F.row_number().over(window))

window = Window.orderBy(F.asc("ave_rank"))

joined_rankings = joined_rankings.withColumn("Growth Potential Ranking", F.row_number().over(window))

joined_rankings = joined_rankings.withColumn(
    "composite_ranking",
    (F.col("Takings Rank") + F.col("Loss Potential Ranking") + F.col("Growth Potential Ranking"))/3
)

window = Window.orderBy(F.asc("composite_ranking"))

joined_rankings = joined_rankings.withColumn("Final Ranking", F.row_number().over(window))

joined_rankings= joined_rankings.select(
    "merchant_abn", "merchant_name", "segment", "Takings Rank", 
    "Loss Potential Ranking", "Growth Potential Ranking", "Final Ranking")

joined_rankings.show(100, truncate=False)

joined_rankings.write.csv("../data/final_ranks/final_merchant_rankings.csv", header=True, mode="overwrite")

In [ ]:
h_g_l_segment_rank = joined_rankings.filter(joined_rankings['segment'] == 'Home, Garden & Living').orderBy('Final Ranking')

h_g_l_segment_rank.show(10, truncate=False)

h_g_l_segment_rank.write.csv("../data/final_ranks/h_g_l_segment_rank.csv", header=True, mode="overwrite")

In [ ]:
l_h_r_segment_rank = joined_rankings.filter(joined_rankings['segment'] == 'Lifestyle, Health & Recreation').orderBy('Final Ranking')

l_h_r_segment_rank.show(10, truncate=False)

l_h_r_segment_rank.write.csv("../data/final_ranks/l_h_r_segment_rank.csv", header=True, mode="overwrite")

In [ ]:
t_p_s_segment_rank = joined_rankings.filter(joined_rankings['segment'] == 'Technology & Professional Services').orderBy('Final Ranking')

t_p_s_segment_rank.show(10, truncate=False)

t_p_s_segment_rank.write.csv("../data/final_ranks/t_p_s_segment_rank.csv", header=True, mode="overwrite")

In [ ]:
a_m_e_segment_rank = joined_rankings.filter(joined_rankings['segment'] == 'Arts, Media & Entertainment').orderBy('Final Ranking')

a_m_e_segment_rank.show(10, truncate=False)

a_m_e_segment_rank.write.csv("../data/final_ranks/a_m_e_segment_rank.csv", header=True, mode="overwrite")

In [ ]:
f_j_p_segment_rank = joined_rankings.filter(joined_rankings['segment'] == 'Fashion, Jewelry & Personal Goods').orderBy('Final Ranking')

f_j_p_segment_rank.show(10, truncate=False)

f_j_p_segment_rank.write.csv("../data/final_ranks/f_j_p_segment_rank.csv", header=True, mode="overwrite")